# EP09. Long Context vs RAG
## 200K 컨텍스트 시대, RAG가 정말 필요 없을까?

---

## 학습 목표

1. **Needle In A Haystack(NIAH) 테스트**를 직접 구현한다
2. **Long Context와 RAG 방식의 정확도·비용·속도**를 정량적으로 비교한다
3. **LangChain으로 두 가지 파이프라인**을 구축한다
4. **Langfuse로 비용과 지연시간**을 자동 추적한다

---

## AI 코딩 가이드

이 노트북은 Claude나 ChatGPT와 함께 실습하도록 설계되었습니다.

```
추천 프롬프트 예시:

1. "NIAH 테스트에서 needle 위치를 10%, 50%, 90%로 바꿔서 정확도를 비교해줘"
2. "RAG 파이프라인에서 chunk_size를 500, 1000, 2000으로 변경했을 때 성능 차이를 분석해줘"
3. "현재 비용 계산 코드를 Claude Sonnet 모델 단가도 포함하도록 확장해줘"
4. "레이더 차트에 '최신성'과 '확장성' 축을 추가해줘"
```

---

**사용 모델:**
- `claude-haiku-4-5-20251001` (Anthropic)
- `gpt-4o-mini` (OpenAI)

**필요한 API 키 (.env 파일):**
```
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
```

---
## 섹션 1. 환경 설정

필요한 패키지를 설치합니다. `uv`를 사용하면 더 빠르게 설치할 수 있습니다.

In [ ]:
# uv pip install (권장)
# !uv pip install langchain langchain-anthropic langchain-openai langchain-community \
#                 chromadb langfuse python-dotenv tiktoken matplotlib numpy

# pip install (대안)
!pip install -q langchain langchain-anthropic langchain-openai langchain-community \
               chromadb langfuse python-dotenv tiktoken matplotlib numpy

---
## 섹션 2. 라이브러리 로드 및 API 키 설정

`.env` 파일에서 API 키를 로드하고 필요한 모든 라이브러리를 임포트합니다.

In [ ]:
import os
import time
import random
import string
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

import tiktoken
from dotenv import load_dotenv

# LangChain 핵심
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.schema import Document

# Langfuse
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler

# .env 로드
load_dotenv()

# 환경 변수 확인
required_keys = [
    "ANTHROPIC_API_KEY",
    "OPENAI_API_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
]

print("API 키 로드 상태:")
for key in required_keys:
    value = os.environ.get(key, "")
    status = "✅ 로드됨" if value else "❌ 없음"
    masked = value[:8] + "..." if len(value) > 8 else "(미설정)"
    print(f"  {key}: {status} ({masked})")

In [ ]:
# 한글 폰트 설정 (macOS)
matplotlib.rc('font', family='AppleGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

# LLM 클라이언트 초기화
claude_haiku = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    temperature=0,
)

gpt4o_mini = ChatOpenAI(
    model="gpt-4o-mini",
    max_tokens=1024,
    temperature=0,
)

# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("모델 초기화 완료")
print(f"  Claude Haiku: {claude_haiku.model}")
print(f"  GPT-4o-mini: {gpt4o_mini.model_name}")
print(f"  Embeddings: {embeddings.model}")

---
## 섹션 3. NIAH 테스트 - 긴 문서(Haystack) 생성

**Needle In A Haystack(NIAH) 테스트**란:
- 긴 텍스트(haystack) 속에 특정 정보(needle)를 숨기고
- LLM이 그 정보를 정확히 찾아내는지 테스트하는 방법입니다

**실험 변수:**
- 문서 길이: 4K / 16K / 32K 토큰
- Needle 위치: 앞(10%) / 중간(50%) / 뒤(90%)

In [ ]:
# tiktoken 인코더 (토큰 수 계산용)
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """텍스트의 토큰 수를 반환합니다."""
    return len(enc.encode(text))

# 랜덤 한국어 문단 생성
FILLER_SENTENCES = [
    "인공지능 기술은 최근 몇 년간 빠르게 발전하고 있으며, 다양한 산업 분야에 활용되고 있습니다.",
    "머신러닝 모델은 대규모 데이터셋으로 학습하여 패턴을 인식하고 예측을 수행합니다.",
    "자연어 처리 기술은 텍스트 데이터를 분석하고 이해하는 데 중요한 역할을 합니다.",
    "딥러닝은 신경망 구조를 기반으로 복잡한 문제를 해결하는 강력한 방법론입니다.",
    "클라우드 컴퓨팅은 대규모 데이터 처리와 모델 학습에 필수적인 인프라를 제공합니다.",
    "데이터 전처리는 모델 성능에 큰 영향을 미치는 중요한 단계입니다.",
    "트랜스포머 아키텍처는 현대 대형 언어 모델의 핵심 구조로 자리잡았습니다.",
    "파인튜닝을 통해 사전 학습된 모델을 특정 도메인에 맞게 최적화할 수 있습니다.",
    "강화학습은 에이전트가 환경과 상호작용하며 최적의 정책을 학습하는 방법입니다.",
    "컴퓨터 비전 기술은 이미지와 영상에서 의미 있는 정보를 추출하는 데 사용됩니다.",
    "벡터 데이터베이스는 고차원 임베딩을 효율적으로 저장하고 검색하는 시스템입니다.",
    "프롬프트 엔지니어링은 LLM의 출력 품질을 향상시키는 중요한 기술입니다.",
    "RAG(검색 증강 생성)은 외부 지식을 활용하여 LLM의 응답을 개선하는 방법입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사도를 계산하는 데 사용됩니다.",
    "API를 통해 다양한 AI 서비스를 쉽게 통합하고 활용할 수 있습니다.",
    "멀티모달 모델은 텍스트, 이미지, 오디오 등 다양한 형태의 데이터를 처리합니다.",
    "지식 그래프는 구조화된 정보를 표현하고 추론하는 데 효과적입니다.",
    "에이전트 시스템은 여러 도구를 자율적으로 활용하여 복잡한 작업을 수행합니다.",
]

def generate_haystack(target_tokens: int, seed: int = 42) -> str:
    """목표 토큰 수에 맞는 haystack(긴 텍스트)을 생성합니다."""
    random.seed(seed)
    paragraphs = []
    current_tokens = 0
    
    while current_tokens < target_tokens:
        # 랜덤 문단 생성 (5~10개 문장)
        num_sentences = random.randint(5, 10)
        sentences = random.choices(FILLER_SENTENCES, k=num_sentences)
        paragraph = " ".join(sentences)
        paragraphs.append(paragraph)
        current_tokens += count_tokens(paragraph)
    
    return "\n\n".join(paragraphs)

# 테스트: 4K 토큰 haystack 생성
sample_haystack = generate_haystack(4000)
actual_tokens = count_tokens(sample_haystack)
print(f"생성된 haystack: {actual_tokens:,} 토큰")
print(f"미리보기: {sample_haystack[:200]}...")

In [ ]:
# Needle 정의
NEEDLE = "피자집의 와이파이 비밀번호는 'AI2025SECRET'입니다."
NEEDLE_QUESTION = "피자집의 와이파이 비밀번호는 무엇인가요?"
NEEDLE_ANSWER = "AI2025SECRET"

def insert_needle(haystack: str, needle: str, position_ratio: float) -> str:
    """
    haystack의 특정 위치(position_ratio)에 needle을 삽입합니다.
    
    Args:
        haystack: 긴 텍스트
        needle: 삽입할 특정 정보
        position_ratio: 삽입 위치 비율 (0.0=앞, 0.5=중간, 1.0=끝)
    """
    paragraphs = haystack.split("\n\n")
    insert_idx = int(len(paragraphs) * position_ratio)
    insert_idx = max(0, min(insert_idx, len(paragraphs) - 1))
    
    paragraphs.insert(insert_idx, f"\n[중요 정보] {needle}\n")
    return "\n\n".join(paragraphs)

# 세 가지 위치에 needle 삽입 테스트
for pos_name, pos_ratio in [("앞(10%)", 0.1), ("중간(50%)", 0.5), ("뒤(90%)", 0.9)]:
    doc_with_needle = insert_needle(sample_haystack, NEEDLE, pos_ratio)
    needle_pos = doc_with_needle.find(NEEDLE)
    total_len = len(doc_with_needle)
    actual_pos = needle_pos / total_len * 100
    print(f"위치 {pos_name}: needle이 전체 문서의 {actual_pos:.1f}% 지점에 삽입됨")

---
## 섹션 4. NIAH 실험 - Long Context 방식으로 질의

전체 문서를 LLM 컨텍스트에 그대로 넣고 needle을 찾도록 질의합니다.

**동작 방식:**
```
[전체 문서 (수천~수만 토큰)] + [질문] → LLM → [답변]
```

In [ ]:
def query_long_context(llm, document: str, question: str) -> dict:
    """
    Long Context 방식: 전체 문서를 컨텍스트로 사용하여 질의합니다.
    
    Returns:
        dict: 답변, 소요시간, 입력토큰수, 출력토큰수
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 문서를 읽고 질문에 답하세요. 답변은 핵심 정보만 간결하게 작성하세요."),
        ("human", "문서:\n{document}\n\n질문: {question}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    
    start_time = time.perf_counter()
    answer = chain.invoke({"document": document, "question": question})
    elapsed_time = time.perf_counter() - start_time
    
    # 토큰 수 계산 (tiktoken)
    input_text = f"다음 문서를 읽고 질문에 답하세요.\n\n문서:\n{document}\n\n질문: {question}"
    input_tokens = count_tokens(input_text)
    output_tokens = count_tokens(answer)
    
    return {
        "answer": answer,
        "elapsed_time": elapsed_time,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

def evaluate_answer(answer: str, expected: str) -> bool:
    """답변에 정답이 포함되어 있는지 확인합니다 (Exact Match)."""
    return expected.lower() in answer.lower()

print("Long Context 쿼리 함수 정의 완료")

In [ ]:
# NIAH 실험: Long Context 방식
# 문서 크기: 4K / 8K / 16K 토큰 (비용 절약을 위해 축소)
# Needle 위치: 앞(10%) / 중간(50%) / 뒤(90%)

DOC_SIZES = [(4_000, "4K"), (8_000, "8K"), (16_000, "16K")]
NEEDLE_POSITIONS = [(0.1, "앞(10%)"), (0.5, "중간(50%)"), (0.9, "뒤(90%)")]

lc_results = []

print("=" * 60)
print("Long Context NIAH 실험 시작")
print("=" * 60)

for target_tokens, size_name in DOC_SIZES:
    haystack = generate_haystack(target_tokens)
    
    for pos_ratio, pos_name in NEEDLE_POSITIONS:
        doc_with_needle = insert_needle(haystack, NEEDLE, pos_ratio)
        actual_tokens = count_tokens(doc_with_needle)
        
        print(f"\n문서 크기: {size_name} ({actual_tokens:,} 토큰), Needle 위치: {pos_name}")
        
        # Claude Haiku로 테스트
        claude_result = query_long_context(claude_haiku, doc_with_needle, NEEDLE_QUESTION)
        claude_correct = evaluate_answer(claude_result["answer"], NEEDLE_ANSWER)
        
        print(f"  Claude Haiku: {'✅ 정답' if claude_correct else '❌ 오답'} "
              f"| {claude_result['elapsed_time']:.2f}초 "
              f"| {claude_result['input_tokens']:,} 입력 토큰")
        print(f"  답변: {claude_result['answer'][:100]}")
        
        lc_results.append({
            "model": "claude-haiku",
            "doc_size": size_name,
            "position": pos_name,
            "correct": claude_correct,
            "elapsed_time": claude_result["elapsed_time"],
            "input_tokens": claude_result["input_tokens"],
            "output_tokens": claude_result["output_tokens"],
        })

print("\n" + "=" * 60)
print("Long Context 실험 완료")

---
## 섹션 5. NIAH 실험 - RAG 방식으로 질의

문서를 청크로 분할하고 벡터 검색을 통해 관련 청크만 LLM에 전달합니다.

**동작 방식:**
```
[전체 문서] → [청킹] → [벡터 DB]
                              ↓
[질문] → [임베딩] → [유사도 검색] → [관련 청크 3~5개] → [LLM] → [답변]
```

In [ ]:
def build_rag_pipeline(documents: list[str], chunk_size: int = 500, chunk_overlap: int = 50):
    """
    RAG 파이프라인을 구축합니다.
    
    Args:
        documents: 문서 텍스트 리스트
        chunk_size: 청크 크기 (문자 수)
        chunk_overlap: 청크 간 오버랩 (문자 수)
    
    Returns:
        vectorstore: Chroma 벡터스토어
    """
    # 텍스트 분할
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""],
    )
    
    docs = [Document(page_content=doc) for doc in documents]
    chunks = splitter.split_documents(docs)
    
    print(f"  청킹 완료: {len(chunks)}개 청크 (chunk_size={chunk_size})")
    
    # 벡터스토어 생성 (인메모리 Chroma)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="niah_test",
    )
    
    return vectorstore, chunks

def query_rag(
    llm,
    vectorstore,
    question: str,
    k: int = 5,
) -> dict:
    """
    RAG 방식으로 질의합니다.
    
    Args:
        llm: LLM 모델
        vectorstore: 벡터스토어
        question: 질문
        k: 검색할 청크 수
    
    Returns:
        dict: 답변, 소요시간, 검색된 청크, 토큰 수
    """
    # 유사도 검색
    retrieval_start = time.perf_counter()
    retrieved_docs = vectorstore.similarity_search(question, k=k)
    retrieval_time = time.perf_counter() - retrieval_start
    
    # 검색된 청크를 컨텍스트로 조합
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # LLM 질의
    prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 컨텍스트를 참고하여 질문에 답하세요. 답변은 핵심 정보만 간결하게 작성하세요."),
        ("human", "컨텍스트:\n{context}\n\n질문: {question}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    
    llm_start = time.perf_counter()
    answer = chain.invoke({"context": context, "question": question})
    llm_time = time.perf_counter() - llm_start
    
    total_time = retrieval_time + llm_time
    
    # 토큰 수 계산
    input_text = f"다음 컨텍스트를 참고하여 질문에 답하세요.\n\n컨텍스트:\n{context}\n\n질문: {question}"
    input_tokens = count_tokens(input_text)
    output_tokens = count_tokens(answer)
    
    return {
        "answer": answer,
        "elapsed_time": total_time,
        "retrieval_time": retrieval_time,
        "llm_time": llm_time,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "retrieved_chunks": len(retrieved_docs),
    }

print("RAG 파이프라인 함수 정의 완료")

In [ ]:
# NIAH 실험: RAG 방식
rag_results = []

print("=" * 60)
print("RAG NIAH 실험 시작")
print("=" * 60)

for target_tokens, size_name in DOC_SIZES:
    haystack = generate_haystack(target_tokens)
    
    for pos_ratio, pos_name in NEEDLE_POSITIONS:
        doc_with_needle = insert_needle(haystack, NEEDLE, pos_ratio)
        
        print(f"\n문서 크기: {size_name}, Needle 위치: {pos_name}")
        
        # RAG 파이프라인 구축
        vectorstore, chunks = build_rag_pipeline([doc_with_needle], chunk_size=300)
        
        # Claude Haiku로 RAG 테스트
        rag_result = query_rag(claude_haiku, vectorstore, NEEDLE_QUESTION, k=5)
        rag_correct = evaluate_answer(rag_result["answer"], NEEDLE_ANSWER)
        
        print(f"  RAG (Claude Haiku): {'✅ 정답' if rag_correct else '❌ 오답'} "
              f"| {rag_result['elapsed_time']:.2f}초 "
              f"| {rag_result['input_tokens']:,} 입력 토큰 "
              f"| 검색 청크: {rag_result['retrieved_chunks']}개")
        print(f"  답변: {rag_result['answer'][:100]}")
        
        rag_results.append({
            "model": "claude-haiku",
            "doc_size": size_name,
            "position": pos_name,
            "correct": rag_correct,
            "elapsed_time": rag_result["elapsed_time"],
            "input_tokens": rag_result["input_tokens"],
            "output_tokens": rag_result["output_tokens"],
        })
        
        # 메모리 정리
        vectorstore.delete_collection()

print("\n" + "=" * 60)
print("RAG 실험 완료")

---
## 섹션 6. RAG 파이프라인 구축 (LangChain)

**[LangChain RAG 파이프라인 아키텍처]**
```mermaid
flowchart LR
    subgraph Indexing ["📚 인덱싱 파이프라인 (오프라인)"]
        direction LR
        D(/"📄 문서 (Documents)"\):::doc --> S("✂️ 청킹 (TextSplitter)"):::process
        S --> E("🧠 임베딩 (Embeddings)"):::model
        E --> V[("🗄️ 벡터 DB (ChromaDB)")]:::db
    end
    subgraph Retrieval ["🔍 검색 및 생성 파이프라인 (온라인)"]
        direction LR
        Q(("👤 사용자 질의")):::query --> QE("🧠 쿼리 임베딩"):::model
        QE --> VS("🔍 유사도 검색 (top-k)"):::search
        V -.-> VS
        VS --> P{"✨ 질의 + 청크 조합"}:::merge
        P --> LLM("🤖 LLM (GPT/Claude)"):::llm
        LLM --> A(["🎯 최종 답변"]):::ans
    end
    classDef doc fill:#eceff1,stroke:#90a4ae,stroke-width:2px,color:#000
    classDef process fill:#bbdefb,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef model fill:#e1bee7,stroke:#8e24aa,stroke-width:2px,color:#000
    classDef db fill:#ffcc80,stroke:#f57c00,stroke-width:2px,color:#000
    classDef query fill:#b2ebf2,stroke:#00acc1,stroke-width:2px,color:#000
    classDef search fill:#ffecb3,stroke:#ffb300,stroke-width:2px,color:#000
    classDef merge fill:#f8bbd0,stroke:#d81b60,stroke-width:2px,color:#000
    classDef llm fill:#c5cae9,stroke:#3f51b5,stroke-width:2px,color:#000
    classDef ans fill:#c8e6c9,stroke:#43a047,stroke-width:2px,font-weight:bold,color:#000
```

실제 QA 시스템에 사용할 수 있는 완성된 RAG 파이프라인을 구축합니다.

**주요 컴포넌트:**
- `RecursiveCharacterTextSplitter`: 텍스트 분할
- `ChromaDB`: 벡터 저장 및 검색
- `RetrievalQA`: 검색+생성 통합 체인

In [ ]:
# 샘플 문서 준비 (실제로는 PDF, TXT 파일 등을 로드)
SAMPLE_DOCUMENTS = [
    """
    회사 정책 문서 - 제1장 근무 규정
    
    1.1 근무 시간
    정규 근무 시간은 오전 9시부터 오후 6시까지입니다. 점심 시간은 오후 12시부터 1시까지이며,
    이 시간은 근무 시간에서 제외됩니다. 재택근무는 주 2회까지 허용됩니다.
    
    1.2 연차 휴가
    신입 직원은 연 15일의 연차가 주어집니다. 근속 3년 이상은 연 17일, 5년 이상은 연 20일입니다.
    연차는 당해 연도에 사용하지 않으면 다음 연도로 이월되지 않습니다.
    
    1.3 초과 근무
    초과 근무는 사전 승인이 필요합니다. 초과 근무 수당은 시급의 1.5배로 지급됩니다.
    """,
    """
    회사 정책 문서 - 제2장 복리후생
    
    2.1 건강 보험
    회사는 모든 정규직 직원에게 건강보험료의 50%를 지원합니다. 가족 보험의 경우 30%를 지원합니다.
    
    2.2 식비 지원
    월 10만원의 식비가 복지 포인트로 지급됩니다. 복지 포인트는 지정된 제휴 식당과 배달앱에서 사용 가능합니다.
    
    2.3 교육 지원
    연간 200만원까지 업무 관련 교육비를 지원합니다. 단, 수료 후 1년 이내에 퇴사할 경우 환급 대상이 됩니다.
    """,
    """
    회사 정책 문서 - 제3장 IT 보안 정책
    
    3.1 비밀번호 정책
    비밀번호는 최소 12자 이상이어야 하며, 대문자, 소문자, 숫자, 특수문자를 각각 포함해야 합니다.
    비밀번호는 90일마다 변경해야 합니다.
    
    3.2 데이터 보안
    고객 데이터는 암호화하여 저장해야 하며, 외부로 반출 시 반드시 승인을 받아야 합니다.
    USB 등 외부 저장매체 사용은 IT 팀의 승인이 필요합니다.
    
    3.3 원격 접속
    회사 내부 시스템 접근 시 VPN을 사용해야 합니다. VPN 계정 신청은 IT 팀에 문의하세요.
    """,
]

# 완성된 RAG 파이프라인
def create_rag_chain(documents: list[str], llm, chunk_size: int = 500) -> RetrievalQA:
    """
    LangChain RetrievalQA 체인을 생성합니다.
    """
    # 1. 텍스트 분할
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=50,
    )
    docs = [Document(page_content=doc) for doc in documents]
    chunks = splitter.split_documents(docs)
    print(f"청크 수: {len(chunks)}")
    
    # 2. 벡터스토어 생성
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="company_policy",
    )
    
    # 3. RetrievalQA 체인 생성
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True,
    )
    
    return qa_chain, vectorstore

# RAG 체인 생성
print("RAG 파이프라인 구축 중...")
rag_chain, policy_vectorstore = create_rag_chain(SAMPLE_DOCUMENTS, claude_haiku)

# 테스트 질의
test_questions = [
    "연차 휴가는 몇 일인가요?",
    "초과 근무 수당은 어떻게 계산되나요?",
    "비밀번호는 얼마나 자주 변경해야 하나요?",
]

print("\n=== RAG 파이프라인 테스트 ===")
for q in test_questions:
    result = rag_chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")

---
## 섹션 7. Stuff-All-In 파이프라인 구축 (LangChain)

**[Stuff-All-In 파이프라인 아키텍처]**
```mermaid
flowchart LR
    subgraph Stuffing ["📥 Stuff-All-In 아키텍처"]
        direction LR
        D(/"📄 전체 문서 (Raw Text)"\):::doc --> T("🔢 토큰 수 확인 (tiktoken)"):::check
        T --> C{"한도 초과?"}:::decide
        C -->|"No"| P{"프롬프트 전체 삽입"}:::insert
        C -->|"Yes"| TR("✂️ 자르기 (Truncation)
또는 Map-Reduce"):::process
        P --> LLM("🤖 대용량 LLM 호출"):::llm
        TR --> LLM
        LLM --> A(["🎯 답변"]):::ans
    end
    classDef doc fill:#eceff1,stroke:#90a4ae,stroke-width:2px,color:#000
    classDef check fill:#e1bee7,stroke:#8e24aa,stroke-width:2px,color:#000
    classDef decide fill:#ffcc80,stroke:#f57c00,stroke-width:2px,font-weight:bold,color:#000
    classDef insert fill:#bbdefb,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef process fill:#ffcdd2,stroke:#e53935,stroke-width:2px,color:#000
    classDef llm fill:#c5cae9,stroke:#3f51b5,stroke-width:2px,color:#000
    classDef ans fill:#c8e6c9,stroke:#43a047,stroke-width:2px,font-weight:bold,color:#000
```

모든 문서를 하나의 프롬프트에 직접 삽입하는 방식입니다.

**주의:** 문서가 길 경우 컨텍스트 한도를 초과할 수 있으므로 `tiktoken`으로 토큰 수를 먼저 확인합니다.

In [ ]:
# 모델별 컨텍스트 한도
CONTEXT_LIMITS = {
    "claude-haiku-4-5-20251001": 200_000,
    "gpt-4o-mini": 128_000,
    "gpt-4o": 128_000,
}

def create_stuff_chain(documents: list[str], llm, model_name: str):
    """
    Stuff-All-In 방식: 모든 문서를 단일 프롬프트에 삽입합니다.
    """
    # 전체 문서 결합
    full_context = "\n\n---\n\n".join(documents)
    total_tokens = count_tokens(full_context)
    limit = CONTEXT_LIMITS.get(model_name, 128_000)
    
    print(f"전체 문서 토큰 수: {total_tokens:,} / 한도: {limit:,}")
    
    if total_tokens > limit * 0.8:  # 80% 초과 시 경고
        print(f"⚠️  경고: 컨텍스트 한도의 {total_tokens/limit*100:.1f}%를 사용합니다.")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 문서들을 참고하여 질문에 답하세요. 정확하고 간결하게 답변해주세요."),
        ("human", "문서:\n{context}\n\n질문: {question}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    
    return chain, full_context, total_tokens

def query_stuff_chain(chain, context: str, question: str) -> dict:
    """
    Stuff-All-In 방식으로 질의합니다.
    """
    input_text = f"{context}\n\n{question}"
    input_tokens = count_tokens(input_text)
    
    start_time = time.perf_counter()
    answer = chain.invoke({"context": context, "question": question})
    elapsed_time = time.perf_counter() - start_time
    
    output_tokens = count_tokens(answer)
    
    return {
        "answer": answer,
        "elapsed_time": elapsed_time,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

# Stuff 체인 생성 및 테스트
print("Stuff-All-In 파이프라인 구축 중...")
stuff_chain, full_context, total_tokens = create_stuff_chain(
    SAMPLE_DOCUMENTS, claude_haiku, "claude-haiku-4-5-20251001"
)

print("\n=== Stuff-All-In 테스트 ===")
for q in test_questions:
    result = query_stuff_chain(stuff_chain, full_context, q)
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    print(f"   (입력 {result['input_tokens']:,} 토큰, {result['elapsed_time']:.2f}초)")

---
## 섹션 8. 정확도 측정

RAG와 Long Context 방식의 정확도를 **Exact Match** 기반으로 측정합니다.

**평가 방법:**
- `Exact Match`: 답변에 정답 키워드가 포함되어 있는가
- 여러 질문에 대한 평균 정확도로 측정

In [ ]:
# 평가용 Q&A 데이터셋
QA_DATASET = [
    {"question": "연차 휴가는 신입직원 기준 몇 일인가요?", "answer": "15"},
    {"question": "재택근무는 주 몇 회까지 허용되나요?", "answer": "2"},
    {"question": "초과 근무 수당은 시급의 몇 배인가요?", "answer": "1.5"},
    {"question": "건강보험료 지원 비율은 몇 %인가요?", "answer": "50"},
    {"question": "월 식비 지원 금액은?", "answer": "10만원"},
    {"question": "연간 교육비 지원 한도는?", "answer": "200만원"},
    {"question": "비밀번호는 몇 자 이상이어야 하나요?", "answer": "12"},
    {"question": "비밀번호 변경 주기는?", "answer": "90일"},
    {"question": "원격 접속 시 무엇을 사용해야 하나요?", "answer": "VPN"},
    {"question": "근속 5년 이상의 연차는 몇 일인가요?", "answer": "20"},
]

def evaluate_pipeline(pipeline_fn, questions_answers: list[dict]) -> dict:
    """
    파이프라인의 정확도, 평균 속도, 평균 비용을 측정합니다.
    """
    results = []
    
    for qa in questions_answers:
        result = pipeline_fn(qa["question"])
        correct = evaluate_answer(result["answer"], qa["answer"])
        result["correct"] = correct
        result["question"] = qa["question"]
        result["expected"] = qa["answer"]
        results.append(result)
    
    accuracy = sum(r["correct"] for r in results) / len(results)
    avg_time = sum(r["elapsed_time"] for r in results) / len(results)
    avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
    avg_output_tokens = sum(r["output_tokens"] for r in results) / len(results)
    
    return {
        "accuracy": accuracy,
        "avg_time": avg_time,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "details": results,
    }

# RAG 파이프라인 평가 함수
def rag_query_fn(question: str) -> dict:
    result = query_rag(claude_haiku, policy_vectorstore, question, k=3)
    return result

# Stuff 파이프라인 평가 함수
def stuff_query_fn(question: str) -> dict:
    result = query_stuff_chain(stuff_chain, full_context, question)
    return result

print("정확도 평가 시작...")
print("\n[1] RAG 파이프라인 평가")
rag_eval = evaluate_pipeline(rag_query_fn, QA_DATASET)

print("\n[2] Stuff-All-In 파이프라인 평가")
stuff_eval = evaluate_pipeline(stuff_query_fn, QA_DATASET)

print("\n=== 정확도 평가 결과 ===")
print(f"{'방식':<20} {'정확도':>8} {'평균 속도':>12} {'평균 입력 토큰':>16}")
print("-" * 60)
print(f"{'RAG (Claude Haiku)':<20} {rag_eval['accuracy']:>7.1%} "
      f"{rag_eval['avg_time']:>11.2f}초 {rag_eval['avg_input_tokens']:>15,.0f}")
print(f"{'Stuff (Claude Haiku)':<20} {stuff_eval['accuracy']:>7.1%} "
      f"{stuff_eval['avg_time']:>11.2f}초 {stuff_eval['avg_input_tokens']:>15,.0f}")

---
## 섹션 9. 비용 계산

`tiktoken`으로 토큰 수를 계산하고 실제 API 단가를 적용하여 비용을 산출합니다.

In [ ]:
# 모델별 API 단가 (USD per 1M tokens, 2024년 기준)
PRICING = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
    "claude-3-5-haiku-20241022": {"input": 0.80, "output": 4.00},
    "claude-3-5-sonnet-20241022": {"input": 3.00, "output": 15.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "text-embedding-3-small": {"input": 0.02, "output": 0.0},
}

def calculate_cost(
    input_tokens: float,
    output_tokens: float,
    model: str,
    num_queries: int = 1,
) -> dict:
    """
    토큰 수와 단가를 기반으로 비용을 계산합니다.
    
    Args:
        input_tokens: 총 입력 토큰 수
        output_tokens: 총 출력 토큰 수
        model: 모델 이름
        num_queries: 쿼리 횟수
    
    Returns:
        dict: 비용 상세 내역 (USD)
    """
    pricing = PRICING.get(model, {"input": 0.0, "output": 0.0})
    
    total_input_tokens = input_tokens * num_queries
    total_output_tokens = output_tokens * num_queries
    
    input_cost = (total_input_tokens / 1_000_000) * pricing["input"]
    output_cost = (total_output_tokens / 1_000_000) * pricing["output"]
    total_cost = input_cost + output_cost
    
    return {
        "input_tokens_total": total_input_tokens,
        "output_tokens_total": total_output_tokens,
        "input_cost_usd": input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": total_cost,
        "total_cost_krw": total_cost * 1380,  # 환율 적용
    }

# 비용 비교: 1일 500회 쿼리 기준
DAILY_QUERIES = 500
MODEL = "claude-haiku-4-5-20251001"

rag_cost = calculate_cost(
    rag_eval['avg_input_tokens'],
    rag_eval['avg_output_tokens'],
    MODEL,
    num_queries=DAILY_QUERIES,
)

stuff_cost = calculate_cost(
    stuff_eval['avg_input_tokens'],
    stuff_eval['avg_output_tokens'],
    MODEL,
    num_queries=DAILY_QUERIES,
)

print(f"=== 비용 비교 (1일 {DAILY_QUERIES:,}회 쿼리 기준, 모델: {MODEL}) ===")
print(f"\n{'항목':<25} {'RAG':>15} {'Stuff-All-In':>15}")
print("-" * 60)
print(f"{'쿼리당 입력 토큰':<25} {rag_eval['avg_input_tokens']:>14,.0f} {stuff_eval['avg_input_tokens']:>14,.0f}")
print(f"{'쿼리당 출력 토큰':<25} {rag_eval['avg_output_tokens']:>14,.0f} {stuff_eval['avg_output_tokens']:>14,.0f}")
print(f"{'일일 입력 비용 ($)':<25} ${rag_cost['input_cost_usd']:>13.4f} ${stuff_cost['input_cost_usd']:>13.4f}")
print(f"{'일일 출력 비용 ($)':<25} ${rag_cost['output_cost_usd']:>13.4f} ${stuff_cost['output_cost_usd']:>13.4f}")
print(f"{'일일 총 비용 ($)':<25} ${rag_cost['total_cost_usd']:>13.4f} ${stuff_cost['total_cost_usd']:>13.4f}")
print(f"{'일일 총 비용 (₩)':<25} ₩{rag_cost['total_cost_krw']:>13,.0f} ₩{stuff_cost['total_cost_krw']:>13,.0f}")

savings_ratio = (1 - rag_cost['total_cost_usd'] / stuff_cost['total_cost_usd']) * 100 if stuff_cost['total_cost_usd'] > 0 else 0
print(f"\n💡 RAG 사용 시 비용 절감: {savings_ratio:.1f}%")

---
## 섹션 10. 속도 측정

`time.perf_counter()`로 각 파이프라인의 응답 시간을 측정합니다.

In [ ]:
# 속도 측정: 동일 질문을 5회 반복
SPEED_TEST_QUESTION = "비밀번호 정책에 대해 설명해주세요."
SPEED_TEST_RUNS = 3  # 비용 절약을 위해 3회로 제한

def benchmark_speed(pipeline_fn, question: str, runs: int = 3) -> dict:
    """
    파이프라인의 응답 속도를 벤치마크합니다.
    """
    times = []
    for i in range(runs):
        result = pipeline_fn(question)
        times.append(result["elapsed_time"])
        print(f"  실행 {i+1}: {result['elapsed_time']:.3f}초")
    
    return {
        "times": times,
        "mean": np.mean(times),
        "std": np.std(times),
        "min": np.min(times),
        "max": np.max(times),
    }

print("속도 벤치마크 시작 (각 3회 반복)...")

print("\n[RAG 속도 측정]")
rag_speed = benchmark_speed(rag_query_fn, SPEED_TEST_QUESTION, SPEED_TEST_RUNS)

print("\n[Stuff-All-In 속도 측정]")
stuff_speed = benchmark_speed(stuff_query_fn, SPEED_TEST_QUESTION, SPEED_TEST_RUNS)

print("\n=== 속도 벤치마크 결과 ===")
print(f"{'방식':<20} {'평균(초)':>10} {'표준편차':>10} {'최소(초)':>10} {'최대(초)':>10}")
print("-" * 65)
print(f"{'RAG':<20} {rag_speed['mean']:>10.3f} {rag_speed['std']:>10.3f} "
      f"{rag_speed['min']:>10.3f} {rag_speed['max']:>10.3f}")
print(f"{'Stuff-All-In':<20} {stuff_speed['mean']:>10.3f} {stuff_speed['std']:>10.3f} "
      f"{stuff_speed['min']:>10.3f} {stuff_speed['max']:>10.3f}")

speedup = stuff_speed['mean'] / rag_speed['mean'] if rag_speed['mean'] > 0 else 1
print(f"\n💡 RAG가 {speedup:.1f}배 빠름")

---
## 섹션 11. 결과 시각화

`matplotlib`으로 비교 결과를 시각화합니다.

**시각화 내용:**
1. 막대그래프: 정확도, 속도, 비용 비교
2. NIAH 히트맵: 위치별 정확도
3. 레이더 차트: 종합 성능 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('RAG vs Long Context (Stuff-All-In) 비교', fontsize=16, fontweight='bold')

colors = ['#2196F3', '#FF5722']  # 파랑: RAG, 빨강: Stuff
labels = ['RAG', 'Stuff-All-In']

# --- 그래프 1: 정확도 비교 ---
ax1 = axes[0]
accuracies = [rag_eval['accuracy'] * 100, stuff_eval['accuracy'] * 100]
bars = ax1.bar(labels, accuracies, color=colors, width=0.5, edgecolor='white', linewidth=2)
ax1.set_title('정확도 비교 (%)', fontsize=13, fontweight='bold')
ax1.set_ylabel('정확도 (%)')
ax1.set_ylim(0, 110)
ax1.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='기준선 (80%)')
ax1.legend(fontsize=9)
for bar, val in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

# --- 그래프 2: 속도 비교 ---
ax2 = axes[1]
speeds = [rag_speed['mean'], stuff_speed['mean']]
speed_errors = [rag_speed['std'], stuff_speed['std']]
bars2 = ax2.bar(labels, speeds, color=colors, width=0.5,
                yerr=speed_errors, capsize=8, edgecolor='white', linewidth=2)
ax2.set_title('응답 속도 비교 (초)', fontsize=13, fontweight='bold')
ax2.set_ylabel('평균 응답 시간 (초)')
for bar, val in zip(bars2, speeds):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
             f'{val:.2f}초', ha='center', va='bottom', fontweight='bold', fontsize=12)

# --- 그래프 3: 비용 비교 ---
ax3 = axes[2]
costs = [rag_cost['total_cost_usd'], stuff_cost['total_cost_usd']]
bars3 = ax3.bar(labels, costs, color=colors, width=0.5, edgecolor='white', linewidth=2)
ax3.set_title(f'일일 비용 비교 (USD, {DAILY_QUERIES}회)', fontsize=13, fontweight='bold')
ax3.set_ylabel('일일 비용 (USD)')
for bar, val in zip(bars3, costs):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
             f'${val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('/tmp/rag_vs_lc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장 완료: /tmp/rag_vs_lc_comparison.png")

In [ ]:
# NIAH 실험 결과 시각화: 위치별 정확도 비교
if lc_results and rag_results:
    # 데이터 정리
    positions = [pos for _, pos in NEEDLE_POSITIONS]
    sizes = [size for _, size in DOC_SIZES]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('NIAH 실험: 위치별 정확도 (Long Context vs RAG)', fontsize=14, fontweight='bold')
    
    for ax_idx, (results, title) in enumerate([
        (lc_results, 'Long Context 방식'),
        (rag_results, 'RAG 방식')
    ]):
        ax = axes[ax_idx]
        
        # 위치 × 크기 히트맵 데이터
        heatmap_data = np.zeros((len(positions), len(sizes)))
        
        for r in results:
            if r['position'] in positions and r['doc_size'] in sizes:
                pos_idx = positions.index(r['position'])
                size_idx = sizes.index(r['doc_size'])
                heatmap_data[pos_idx][size_idx] = 1 if r['correct'] else 0
        
        im = ax.imshow(heatmap_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(len(sizes)))
        ax.set_xticklabels(sizes)
        ax.set_yticks(range(len(positions)))
        ax.set_yticklabels(positions)
        ax.set_xlabel('문서 크기')
        ax.set_ylabel('Needle 위치')
        ax.set_title(title, fontsize=12, fontweight='bold')
        
        for i in range(len(positions)):
            for j in range(len(sizes)):
                val = heatmap_data[i][j]
                text = '✓' if val == 1 else '✗'
                ax.text(j, i, text, ha='center', va='center', fontsize=16,
                        color='white' if val < 0.5 else 'black')
        
        plt.colorbar(im, ax=ax, label='정답률')
    
    plt.tight_layout()
    plt.savefig('/tmp/niah_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("NIAH 히트맵 저장 완료: /tmp/niah_heatmap.png")
else:
    print("NIAH 실험 결과 없음 - 섹션 4, 5를 먼저 실행하세요")

In [ ]:
# 레이더 차트: 종합 성능 비교
categories = ['비용 효율', '정확도', '응답 속도', '구현 편의성', '확장성']
N = len(categories)

# 각 항목을 0-10 점수로 환산
# 비용: RAG가 유리 (절감율에 비례), 속도: RAG가 유리
cost_efficiency_ratio = min(savings_ratio / 100, 1.0) if stuff_cost['total_cost_usd'] > 0 else 0.5
speed_ratio = min((speedup - 1) / 5, 1.0)  # 최대 6배 빠름 기준

rag_scores = [
    9.0 * cost_efficiency_ratio + 0.5,       # 비용 효율
    rag_eval['accuracy'] * 10,               # 정확도
    min(9.5, 5 + speed_ratio * 4),           # 응답 속도
    4.0,                                      # 구현 편의성 (복잡)
    9.5,                                      # 확장성 (우수)
]

stuff_scores = [
    10 - 9.0 * cost_efficiency_ratio,        # 비용 효율
    stuff_eval['accuracy'] * 10,             # 정확도
    max(1.0, 5 - speed_ratio * 3),           # 응답 속도
    9.0,                                      # 구현 편의성 (단순)
    3.0,                                      # 확장성 (제한적)
]

# 레이더 차트 그리기
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # 닫기

fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(polar=True))

rag_plot = rag_scores + rag_scores[:1]
stuff_plot = stuff_scores + stuff_scores[:1]

ax.plot(angles, rag_plot, 'o-', linewidth=2, color='#2196F3', label='RAG')
ax.fill(angles, rag_plot, alpha=0.25, color='#2196F3')

ax.plot(angles, stuff_plot, 'o-', linewidth=2, color='#FF5722', label='Stuff-All-In')
ax.fill(angles, stuff_plot, alpha=0.25, color='#FF5722')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 10)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=8)
ax.grid(color='gray', linestyle='--', alpha=0.3)
ax.set_title('RAG vs Stuff-All-In 종합 비교\n(레이더 차트)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=12)

plt.tight_layout()
plt.savefig('/tmp/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("레이더 차트 저장 완료: /tmp/radar_chart.png")

---
## 섹션 12. Langfuse 비용 추적 통합

**Langfuse**는 LLM 애플리케이션의 비용, 지연시간, 정확도를 자동으로 추적하는 오픈소스 플랫폼입니다.

**Langfuse Python SDK v3 (OpenTelemetry 기반):**
- `CallbackHandler`: LangChain과 통합
- 대시보드에서 토큰 사용량, 비용, 레이턴시 자동 시각화
- 실험(Experiment)별 성능 비교 가능

In [ ]:
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler

# Langfuse 콜백 핸들러 초기화
langfuse_handler = LangfuseCallbackHandler(
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
    host="https://cloud.langfuse.com",
)

print("Langfuse 핸들러 초기화 완료")
print(f"  Public Key: {os.environ.get('LANGFUSE_PUBLIC_KEY', '(미설정)')[:12]}...")

In [ ]:
# Langfuse 추적이 적용된 RAG 파이프라인
def query_rag_with_langfuse(
    llm,
    vectorstore,
    question: str,
    experiment_name: str = "rag-vs-lc",
    run_name: str = "rag",
    k: int = 5,
) -> dict:
    """
    Langfuse 추적이 포함된 RAG 질의입니다.
    """
    # 세션별 핸들러 (실험 분리)
    session_handler = LangfuseCallbackHandler(
        public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
        secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
        host="https://cloud.langfuse.com",
        session_id=f"{experiment_name}-{run_name}",
        trace_name=f"{run_name}_query",
    )
    
    # 유사도 검색
    retrieved_docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 컨텍스트를 참고하여 질문에 답하세요."),
        ("human", "컨텍스트:\n{context}\n\n질문: {question}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    
    start_time = time.perf_counter()
    # config에 콜백 전달 → Langfuse 자동 추적
    answer = chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [session_handler]},
    )
    elapsed_time = time.perf_counter() - start_time
    
    input_tokens = count_tokens(f"{context}\n\n{question}")
    output_tokens = count_tokens(answer)
    
    return {
        "answer": answer,
        "elapsed_time": elapsed_time,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

# Langfuse 추적이 적용된 Stuff-All-In
def query_stuff_with_langfuse(
    llm,
    context: str,
    question: str,
    experiment_name: str = "rag-vs-lc",
    run_name: str = "stuff",
) -> dict:
    """
    Langfuse 추적이 포함된 Stuff-All-In 질의입니다.
    """
    session_handler = LangfuseCallbackHandler(
        public_key=os.environ.get("LANGFUSE_PUBLIC_KEY", ""),
        secret_key=os.environ.get("LANGFUSE_SECRET_KEY", ""),
        host="https://cloud.langfuse.com",
        session_id=f"{experiment_name}-{run_name}",
        trace_name=f"{run_name}_query",
    )
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "다음 문서를 참고하여 질문에 답하세요."),
        ("human", "문서:\n{context}\n\n질문: {question}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    
    start_time = time.perf_counter()
    answer = chain.invoke(
        {"context": context, "question": question},
        config={"callbacks": [session_handler]},
    )
    elapsed_time = time.perf_counter() - start_time
    
    input_tokens = count_tokens(f"{context}\n\n{question}")
    output_tokens = count_tokens(answer)
    
    return {
        "answer": answer,
        "elapsed_time": elapsed_time,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

print("Langfuse 통합 함수 정의 완료")

In [ ]:
# Langfuse 추적으로 실험 실행
# Langfuse 대시보드(https://cloud.langfuse.com)에서 결과를 확인하세요

LANGFUSE_ENABLED = bool(
    os.environ.get("LANGFUSE_PUBLIC_KEY") and 
    os.environ.get("LANGFUSE_SECRET_KEY")
)

if LANGFUSE_ENABLED:
    print("=== Langfuse 추적 실험 시작 ===")
    
    test_q = "비밀번호는 얼마나 자주 변경해야 하나요?"
    
    print("\n[1] RAG with Langfuse")
    rag_lf_result = query_rag_with_langfuse(
        claude_haiku, policy_vectorstore, test_q,
        experiment_name="ep09-comparison", run_name="rag"
    )
    print(f"  답변: {rag_lf_result['answer']}")
    print(f"  시간: {rag_lf_result['elapsed_time']:.2f}초")
    
    print("\n[2] Stuff-All-In with Langfuse")
    stuff_lf_result = query_stuff_with_langfuse(
        claude_haiku, full_context, test_q,
        experiment_name="ep09-comparison", run_name="stuff"
    )
    print(f"  답변: {stuff_lf_result['answer']}")
    print(f"  시간: {stuff_lf_result['elapsed_time']:.2f}초")
    
    print("\n✅ Langfuse 대시보드에서 확인: https://cloud.langfuse.com")
    print("   - 각 세션의 토큰 사용량 및 비용")
    print("   - 응답 시간 및 레이턴시 분포")
    print("   - 세션 'ep09-comparison-rag' vs 'ep09-comparison-stuff'")
else:
    print("⚠️  Langfuse 키가 설정되지 않았습니다.")
    print("  .env 파일에 LANGFUSE_PUBLIC_KEY와 LANGFUSE_SECRET_KEY를 추가하세요.")
    print("  무료 계정: https://cloud.langfuse.com")

---
## 섹션 13. Exercise

지금까지 배운 내용을 응용하는 실습 문제입니다.

### Exercise 1: NIAH 실험 확장

**목표:** GPT-4o-mini를 추가하여 Claude Haiku vs GPT-4o-mini 모델 간 NIAH 정확도를 비교하세요.

**TODO:**
1. `gpt4o_mini` 모델로 `query_long_context()` 실행
2. 결과를 `lc_results`와 같은 형식으로 저장
3. 두 모델의 정확도를 나란히 비교하는 막대그래프 작성

**힌트:**
```python
# GPT-4o-mini로 테스트
gpt_result = query_long_context(gpt4o_mini, doc_with_needle, NEEDLE_QUESTION)
gpt_correct = evaluate_answer(gpt_result["answer"], NEEDLE_ANSWER)
```

In [ ]:
# Exercise 1: GPT-4o-mini NIAH 실험
# 아래 코드를 완성하세요

gpt_lc_results = []

# TODO: DOC_SIZES와 NEEDLE_POSITIONS를 순회하며
# gpt4o_mini 모델로 Long Context 실험을 수행하고
# gpt_lc_results에 결과를 저장하세요

# --- 여기에 코드를 작성하세요 ---


# 완성 후: Claude Haiku vs GPT-4o-mini 비교 출력
if gpt_lc_results:
    claude_acc = sum(r['correct'] for r in lc_results) / len(lc_results) if lc_results else 0
    gpt_acc = sum(r['correct'] for r in gpt_lc_results) / len(gpt_lc_results)
    
    print(f"Claude Haiku 정확도: {claude_acc:.1%}")
    print(f"GPT-4o-mini 정확도: {gpt_acc:.1%}")

### Exercise 2: 청크 크기가 RAG 정확도에 미치는 영향

**목표:** `chunk_size`를 200, 500, 1000, 2000으로 변경하면서 RAG 정확도 변화를 관찰하세요.

**TODO:**
1. 각 `chunk_size` 값으로 `build_rag_pipeline()` 실행
2. `QA_DATASET` 전체에 대해 정확도 측정
3. x축: chunk_size, y축: 정확도인 꺾은선 그래프 작성

**예상 결과:**
- 청크가 너무 작으면 문맥이 잘려 정확도 저하
- 청크가 너무 크면 Long Context와 유사해짐
- 최적의 청크 크기 존재

In [ ]:
# Exercise 2: 청크 크기별 RAG 정확도 비교
# 아래 코드를 완성하세요

CHUNK_SIZES = [200, 500, 1000, 2000]
chunk_accuracy_results = {}

# TODO: 각 chunk_size에 대해
# 1. build_rag_pipeline()으로 새 vectorstore 생성
# 2. evaluate_pipeline()으로 정확도 측정
# 3. chunk_accuracy_results에 저장

# --- 여기에 코드를 작성하세요 ---


# 완성 후: 청크 크기별 정확도 그래프
if chunk_accuracy_results:
    sizes = list(chunk_accuracy_results.keys())
    accs = [chunk_accuracy_results[s]['accuracy'] * 100 for s in sizes]
    
    plt.figure(figsize=(8, 5))
    plt.plot(sizes, accs, 'bo-', linewidth=2, markersize=10)
    plt.xlabel('Chunk Size (chars)', fontsize=12)
    plt.ylabel('Accuracy (%)', fontsize=12)
    plt.title('Chunk Size vs RAG Accuracy', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.xticks(sizes)
    
    for x, y in zip(sizes, accs):
        plt.annotate(f'{y:.1f}%', (x, y), textcoords="offset points",
                     xytext=(0, 10), ha='center', fontsize=11)
    
    plt.tight_layout()
    plt.show()

### Exercise 3 (보너스): 실제 PDF 문서로 비교 실험

**목표:** 실제 PDF 파일을 로드하여 RAG vs Long Context 비교 실험을 진행하세요.

**TODO:**
1. `pypdf`로 PDF 로드: `pip install pypdf`
2. 최소 10페이지 이상의 PDF 준비 (예: 회사 규정, 논문, 제품 매뉴얼)
3. 10개 이상의 Q&A 데이터셋 직접 작성
4. RAG vs Stuff 비교 실행 후 결과 보고서 작성
5. Langfuse 대시보드에서 비용 비교 캡처

```python
# PDF 로드 예시
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("your_document.pdf")
pages = loader.load()
full_text = "\n\n".join([page.page_content for page in pages])
print(f"PDF 로드 완료: {len(pages)}페이지, {count_tokens(full_text):,} 토큰")
```

---

**이번 에피소드 핵심 정리:**

| 상황 | 권장 방식 |
|------|----------|
| 문서 < 10K 토큰, 단순 분석 | Long Context |
| 문서 > 10K 토큰, 고빈도 질의 | RAG |
| 비용이 최우선 | RAG |
| 실시간 문서 업데이트 필요 | Long Context |
| 정확한 특정 정보 검색 | RAG |
| 전체 문서 요약/분석 | Long Context 또는 Map-Reduce |

**다음 에피소드:** EP10. 멀티모달 컨텍스트 - 이미지와 텍스트를 함께 활용하는 방법